In [2]:
import sys
import os

# Добавляем корневую директорию проекта в sys.path
sys.path.append(os.path.dirname(os.path.abspath('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/tests')))

In [4]:
from utils.utils import make_semantic_columns_name, analyze_dataset_parallel


/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/.myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import sys
sys.path
sys.path.append('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer')

In [5]:
import pandas as pd

In [6]:
data = pd.read_csv('/home/poddubny/notebooks/poddubnyy/postgraduate/doduo/sample_tables/sample_table2.csv')
data

,Unnamed: 0,Rank,Title (click to view),Studio,Adjusted Gross,Unadjusted Gross,Release
0,0,1,Happy Feet,WB,"$243,694,900","$198,000,317",11/17/06
1,1,2,Mad Max: Fury Road,WB,"$151,682,600","$151,682,620",5/15/15
2,2,3,Babe,Uni.,"$118,435,800","$63,658,910",8/4/95
3,3,4,Mad Max Beyond Thunderdome,WB,"$82,870,200","$36,230,219",7/12/85
4,4,5,Happy Feet Two,WB,"$66,334,700","$64,006,466",11/18/11
5,5,6,The Road Warrior,WB,"$65,368,500","$23,667,907",5/21/82
6,6,7,Babe: Pig in the City,Uni.,"$31,139,400","$18,319,860",11/27/98
7,7,8,Mad Max,Film,"$26,412,600","$8,750,000",3/21/80
8,8,9,Lorenzo's Oil,Uni.,"$14,291,200","$7,286,388",1/1/93


In [8]:
make_semantic_columns_name(data,basedir='../utils/doduo/')

Some weights of BertForMultiOutputClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


embeddings tensor([[[ 0.0215, -0.2349, -0.2795,  ..., -0.0381,  0.0224,  0.1561],
         [-0.6128, -0.1561,  0.1591,  ...,  0.2041,  0.8576,  0.1220],
         [-0.2168,  0.2346, -0.0979,  ..., -0.0279,  1.0006,  0.4070],
         ...,
         [ 0.0410,  0.6460, -0.0938,  ..., -0.1959,  0.1694, -0.8710],
         [ 0.1153,  0.1217, -0.6910,  ...,  0.8314,  0.2589,  0.0414],
         [-0.5674,  0.7541, -0.1648,  ..., -0.2590, -0.2490, -0.4067]]],
       grad_fn=<NativeLayerNormBackward0>)


[('Unnamed: 0', {'rank': np.float32(0.9966775)}),
 ('Rank', {'notes': np.float32(0.84829015)}),
 ('Title (click to view)', {'name': np.float32(0.999723)}),
 ('Studio', {'team': np.float32(0.6192725)}),
 ('Adjusted Gross', {'family': np.float32(0.69604427)}),
 ('Unadjusted Gross', {'result': np.float32(0.7002906)}),
 ('Release', {})]

In [5]:
analyze_dataset_parallel(data)

{'Unnamed: 0': ('int', 0),
 'Rank': ('int', 0),
 'Title (click to view)': ('str', 0),
 'Studio': ('str', 0),
 'Adjusted Gross': ('str', 0),
 'Unadjusted Gross': ('str', 0),
 'Release': ('str', 0)}

In [13]:
import xml.etree.ElementTree as ET
import json

In [27]:
import json
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

data_with_numpy = {
    "float_val": np.float32(1.23),
    "int_val": np.int64(42),
    "array_val": np.array([1.1, 2.2, 3.3], dtype=np.float32)
}

json_output = json.dumps(data_with_numpy, cls=NumpyEncoder)
print(json_output)

{"float_val": 1.2300000190734863, "int_val": 42, "array_val": [1.100000023841858, 2.200000047683716, 3.299999952316284]}


In [52]:
from typing import Callable, Any, Dict
import inspect

def get_kwargs(kwargs: Dict[str,Any],func: Callable) -> Dict[str,Any]:
    sig = inspect.signature(func)
    return {key:value for key,value in kwargs.items() if key in sig.parameters}

In [61]:
p = get_kwargs({'top_k':15,'sdjk':5654},make_semantic_columns_name)
print(','.join({**get_kwargs({'top_k':15,'sdjk':5654},make_semantic_columns_name)}))

top_k


In [62]:
def serialize_table(table: pd.DataFrame,include_data_types: bool = True,include_semantic_types: bool = True,include_examples: bool = True,
                    examples_count: int = 3, description: str = "",**kwargs) -> str:
    table_xml = ET.Element("TABLE")
    data_types = None
    sem_types = None
    if description != '':
        descr = ET.SubElement(table_xml, "DESCRIPTION")
        descr.text = description
    if include_data_types:
        data_types = analyze_dataset_parallel(table,**get_kwargs(kwargs,analyze_dataset_parallel))
    if include_semantic_types:
        sem_types = make_semantic_columns_name(table,**get_kwargs(kwargs,make_semantic_columns_name))
    for col_idx, column_name in enumerate(table.columns):
        head =  ET.SubElement(table_xml, "HEADER")
        name = ET.SubElement(head, "NAME")
        name.text = str(column_name)
        if include_semantic_types:
            sem_t = ET.SubElement(head, "SEMANTIC_TYPE")
            sem_t.text = 'NO TYPE'
            sem_t.text = " ; ".join([" - ".join([type_,str(round(prop,2))]) for type_,prop in sem_types[col_idx][1].items()])
        
        if include_data_types:
            data_t = ET.SubElement(head, "DATA_TYPE")
            print('data_type',column_name,data_types[column_name][0])
            data_t.text = json.dumps(data_types[column_name][0])
            data_t_none = ET.SubElement(head, "HAS_NONE")
            print('data_none',column_name,data_types[column_name][1])
            data_t_none.text = '1' if data_types[column_name][1] else '0'
        if include_examples:
            example = ET.SubElement(head, "EXAMPLES")
            example.text = json.dumps(table[column_name].sample(examples_count).to_list(),cls=NumpyEncoder)
        
    return ET.tostring(table_xml, encoding='unicode')
        

In [63]:
serialize_table(data,threshold=0)

Some weights of BertForMultiOutputClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


{'data': tensor([  101,  1014,  1015,  1016,  1017,  1018,  1019,  1020,  1021,  1022,
          102,   101,  1015,  1016,  1017,  1018,  1019,  1020,  1021,  1022,
         1023,   102,   101,  3407,  2519,  5506,  4098,  1024,  8111,  2346,
        11561,  5506,  4098,  3458,  8505, 26173,  3407,  2519,  2048,  1996,
         2346,  6750, 11561,  1024, 10369,  1999,  1996,  2103,  5506,  4098,
        12484,  1005,  1055,  3514,   102,   101, 25610, 25610,  4895,  2072,
         1012, 25610, 25610, 25610,  4895,  2072,  1012,  2143,  4895,  2072,
         1012,   102,   101,  1002, 22884,  1010,  6353,  2549,  1010,  7706,
         1002, 16528,  1010,  6273,  2475,  1010,  5174,  1002, 12963,  1010,
        24125,  1010,  5385,  1002,  6445,  1010, 28864,  1010,  3263,  1002,
         5764,  1010, 29562,  1010,  6352,   102,   101,  1002, 20003,  1010,
         2199,  1010, 26628,  1002, 16528,  1010,  6273,  2475,  1010, 23612,
         1002,  6191,  1010,  3515,  2620,  1010, 27743

'<TABLE><HEADER><NAME>Unnamed: 0</NAME><SEMANTIC_TYPE>rank - 1.0</SEMANTIC_TYPE><DATA_TYPE>"int"</DATA_TYPE><HAS_NONE>0</HAS_NONE><EXAMPLES>[3, 2, 6]</EXAMPLES></HEADER><HEADER><NAME>Rank</NAME><SEMANTIC_TYPE>notes - 0.85</SEMANTIC_TYPE><DATA_TYPE>"int"</DATA_TYPE><HAS_NONE>0</HAS_NONE><EXAMPLES>[2, 8, 9]</EXAMPLES></HEADER><HEADER><NAME>Title (click to view)</NAME><SEMANTIC_TYPE>name - 1.0</SEMANTIC_TYPE><DATA_TYPE>"str"</DATA_TYPE><HAS_NONE>0</HAS_NONE><EXAMPLES>["Mad Max Beyond Thunderdome", "Babe", "Mad Max"]</EXAMPLES></HEADER><HEADER><NAME>Studio</NAME><SEMANTIC_TYPE>team - 0.62</SEMANTIC_TYPE><DATA_TYPE>"str"</DATA_TYPE><HAS_NONE>0</HAS_NONE><EXAMPLES>["Uni.", "Uni.", "Film"]</EXAMPLES></HEADER><HEADER><NAME>Adjusted Gross</NAME><SEMANTIC_TYPE>family - 0.7</SEMANTIC_TYPE><DATA_TYPE>"str"</DATA_TYPE><HAS_NONE>0</HAS_NONE><EXAMPLES>["$243,694,900", "$14,291,200", "$66,334,700"]</EXAMPLES></HEADER><HEADER><NAME>Unadjusted Gross</NAME><SEMANTIC_TYPE>result - 0.7</SEMANTIC_TYPE><DATA

In [8]:
import xml.etree.ElementTree as ET

# 1. Создание корневого элемента
root = ET.Element("items")

# 2. Создание дочерних элементов
item1 = ET.SubElement(root, "item", name="item1")
item2 = ET.SubElement(root, "item", name="item2")

# 3. Добавление атрибутов
item1.set("description", "Это первый элемент")
item2.set("description", "Это второй элемент")

# 4. Создание дерева
tree = ET.ElementTree(root)
root.text = "Это текст корневого элемента."
# 5. Сохранение в файл
# Вывод в консоль (необязательно)
ET.dump(root)

<items><item name="item1" description="Это первый элемент" /><item name="item2" description="Это второй элемент" /></items>
<Element 'items' at 0x7bbb4c7433d0>


In [12]:
ET.tostring(root, encoding='unicode')

'<items><item name="item1" description="Это первый элемент" /><item name="item2" description="Это второй элемент" /></items>'

In [14]:
import numpy as np

# Создаем пример Series
s = pd.Series(np.random.randn(100))

# Получаем 5 случайных примеров
random_samples = s.sample(5)


In [51]:
import inspect
sig = inspect.signature(make_semantic_columns_name)
'top_k' in sig.parameters

TypeError: argument of type 'Signature' is not iterable

In [17]:
random_samples.to_list()

[-0.31397929732730767,
 0.21675007666847226,
 -0.12281349173531601,
 0.45595927079036647,
 -0.8909385828436621]

In [18]:
df = pd.DataFrame({('Группа 1', 'Столбец А'): [1, 2], ('Группа 1', 'Столбец Б'): [3, 4], ('Группа 2', 'Столбец В'): [5, 6]})

In [21]:
df.columns

MultiIndex([('Группа 1', 'Столбец А'),
            ('Группа 1', 'Столбец Б'),
            ('Группа 2', 'Столбец В')],
           )